In [1]:
from flax import nnx
import jax
import jax.numpy as jnp
import optax
import pandas as pd
from sklearn.metrics import classification_report
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [149]:
def convertDataset(dfs):
  df_concat = pd.concat(dfs, axis=1)
  dataset = jnp.asarray(df_concat.values)
  return dataset

df_ndvi = pd.read_csv("drive/MyDrive/train/NDVI.csv", encoding="windows-1251").drop(columns=["index"])
df_re5 = pd.read_csv("drive/MyDrive/train/B05.csv", encoding="windows-1251").drop(columns=["index"])  # B05 - канал со значениями RedEdge 705nm
df_nir = pd.read_csv("drive/MyDrive/train/B8A.csv", encoding="windows-1251").drop(columns=["index"])  # B8A - канал со значениями NIR
df_swir = pd.read_csv("drive/MyDrive/train/B12.csv", encoding="windows-1251").drop(columns=["index"])  # B12 - канал со значениями SWIR
df_re = pd.read_csv("drive/MyDrive/train/B07.csv", encoding="windows-1251").drop(columns=["index"])  # B07 - канал со значениями RedEdge
df_red = pd.read_csv("drive/MyDrive/train/B04.csv", encoding="windows-1251").drop(columns=["index"])  # B04 - канал со значениями Red
df_blue = pd.read_csv("drive/MyDrive/train/B02.csv", encoding="windows-1251").drop(columns=["index"])  # B04 - канал со значениями Blue
df_green = pd.read_csv("drive/MyDrive/train/B03.csv", encoding="windows-1251").drop(columns=["index"]) # B03 - канал со значениями Green
df_sir = pd.read_csv("drive/MyDrive/train/B11.csv", encoding="windows-1251").drop(columns=["index"]) # B03 - канал со значениями SWIR 11 band
df_re6 = pd.read_csv("drive/MyDrive/train/B06.csv", encoding="windows-1251").drop(columns=["index"])  # B06 - канал со значениями RedEdge 740nm

cul_forward = list(df_ndvi['culture'].unique().tolist())
cul = {item: i for i, item in enumerate(cul_forward)}
answers_y =[]
for item in df_ndvi["culture"]:
  s = [0.0] * 6
  s[cul[item]] = 1.0
  answers_y.append(s)

df_stack = [df_red, df_green, df_blue, df_nir, df_re6, df_sir, df_swir, df_re, df_re5, df_ndvi]
for i in range(len(df_stack)):
  df_stack[i] = df_stack[i].drop(columns=["culture"])
dataset = convertDataset(df_stack)

print(cul_forward)
print(cul)
print(answers_y)
print(dataset.devices())
print(dataset)
print(dataset.shape)

['овощи', 'соя', 'кукуруза', 'залежь', 'многолетние травы', 'зерновые']
{'овощи': 0, 'соя': 1, 'кукуруза': 2, 'залежь': 3, 'многолетние травы': 4, 'зерновые': 5}
[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0,

In [151]:
df_ndvi = pd.read_csv("drive/MyDrive/test/NDVI.csv", sep=";", encoding="windows-1251").drop(columns=["index"])
df_re5 = pd.read_csv("drive/MyDrive/test/B05.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B05 - канал со значениями RedEdge 705nm
df_nir = pd.read_csv("drive/MyDrive/test/B8A.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B8A - канал со значениями NIR
df_swir = pd.read_csv("drive/MyDrive/test/B12.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B12 - канал со значениями SWIR
df_re = pd.read_csv("drive/MyDrive/test/B07.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B07 - канал со значениями RedEdge
df_red = pd.read_csv("drive/MyDrive/test/B04.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B04 - канал со значениями Red
df_blue = pd.read_csv("drive/MyDrive/test/B02.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B02 - канал со значениями Blue
df_green = pd.read_csv("drive/MyDrive/test/B03.csv", sep=";", encoding="windows-1251").drop(columns=["index"]) # B03 - канал со значениями Green
df_sir = pd.read_csv("drive/MyDrive/test/B11.csv", sep=";", encoding="windows-1251").drop(columns=["index"]) # B11 - канал со значениями SWIR 11 band
df_re6 = pd.read_csv("drive/MyDrive/test/B06.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B06 - канал со значениями RedEdge 740nm
df_stack = [df_red, df_green, df_blue, df_nir, df_re6, df_sir, df_swir, df_re, df_re5, df_ndvi]
opendataset = convertDataset(df_stack)

df_ndvi = pd.read_csv("drive/MyDrive/closedtest/NDVI.csv", sep=";", encoding="windows-1251").drop(columns=["index"])
df_re5 = pd.read_csv("drive/MyDrive/closedtest/B05.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B05 - канал со значениями RedEdge 705nm
df_nir = pd.read_csv("drive/MyDrive/closedtest/B8A.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B8A - канал со значениями NIR
df_swir = pd.read_csv("drive/MyDrive/closedtest/B12.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B12 - канал со значениями SWIR
df_re = pd.read_csv("drive/MyDrive/closedtest/B07.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B07 - канал со значениями RedEdge
df_red = pd.read_csv("drive/MyDrive/closedtest/B04.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B04 - канал со значениями Red
df_blue = pd.read_csv("drive/MyDrive/closedtest/B02.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B02 - канал со значениями Blue
df_green = pd.read_csv("drive/MyDrive/closedtest/B03.csv", sep=";", encoding="windows-1251").drop(columns=["index"]) # B03 - канал со значениями Green
df_sir = pd.read_csv("drive/MyDrive/closedtest/B11.csv", sep=";", encoding="windows-1251").drop(columns=["index"]) # B11 - канал со значениями SWIR 11 band
df_re6 = pd.read_csv("drive/MyDrive/closedtest/B06.csv", sep=";", encoding="windows-1251").drop(columns=["index"])  # B06 - канал со значениями RedEdge 740nm
df_stack = [df_red, df_green, df_blue, df_nir, df_re6, df_sir, df_swir, df_re, df_re5, df_ndvi]
closeddataset = convertDataset(df_stack)

In [152]:
class LearnableImputeLayer(nnx.Module):
    def __init__(self, din: int):
        initializer = nnx.initializers.normal()
        w = initializer(jax.random.key(42), (din,), jnp.float32)
        self.impute_values = nnx.Param(w)

    def __call__(self, x):
        # x: (batch, n_features)
        # Create mask where x == sentinel, in this version sentinel excluded and changed to straight NaN check
        mask = jnp.isnan(x)
        # Expand impute_values to broadcast: (1, n_features)
        # Added casting to initial data type
        impute_vals = self.impute_values[None, :].astype(x.dtype)
        # Replace missing entries
        x_imputed = jnp.where(mask, impute_vals, x)
        return x_imputed

class ResidualBlock(nnx.Module):
    def __init__(self, dim, rngs):
        self.linear = nnx.Linear(dim, dim, rngs=rngs)
        self.bn = nnx.BatchNorm(dim, rngs=rngs)

    def __call__(self, x):
        residual = x
        x = self.linear(x)
        x = nnx.leaky_relu(x)
        x = self.bn(x)
        return x + residual  # residual connection

class MainBlock(nnx.Module):
    def __init__(self, din, dout, rngs):
        self.linear = nnx.Linear(din, dout, rngs=rngs)
        self.bn = nnx.BatchNorm(din, rngs=rngs)
        self.dropout = nnx.Dropout(rate=0.1, rngs=rngs)

    def __call__(self, x):
        x = self.bn(x)
        x = self.linear(x)
        x = nnx.leaky_relu(x)
        x = self.dropout(x)
        return x

In [153]:
class MLP(nnx.Module):
    def __init__(self, din: int, dmid: int, dout: int, *, rngs: nnx.Rngs):
        """
        Args:
            din: Input dimension.
            dmid: Hidden layer dimension.
            dout: Output dimension.
            rngs: NNX RNG manager for parameter initialization.
        """
        self.impute = LearnableImputeLayer(din)
        self.block1 = MainBlock(din, dmid, rngs=rngs)
        self.block2 = MainBlock(dmid, 256, rngs=rngs)
        self.block3 = MainBlock(256, dmid, rngs=rngs)
        self.rb = ResidualBlock(dmid, rngs=rngs)
        self.linear4 = nnx.Linear(dmid, dout, rngs=rngs)

    def __call__(self, x: jax.Array):
        """Defines the forward pass."""
        x = self.impute(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.rb(x)
        x = self.linear4(x)
        return x

In [154]:
# Initialize model and optimizer
model = MLP(din=260, dmid=512, dout=6, rngs=nnx.Rngs(42))
optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=0.0025), wrt=nnx.Param)
model.train()

@nnx.jit
def train_step(model, optimizer, batch_x, batch_y):
    def combined_loss(model):
        pred_y = model(batch_x)
        loss_quad = jnp.mean((batch_y - pred_y) ** 2)
        loss_cross = jnp.mean(optax.softmax_cross_entropy(pred_y, batch_y))
        return (loss_quad + loss_cross) / 2

    loss, grads = nnx.value_and_grad(combined_loss)(model)
    optimizer.update(model, grads)
    return loss

batch_x = dataset
batch_y = jnp.array(answers_y)

for step in range(310):
    loss = train_step(model, optimizer, batch_x, batch_y)
    print(f"Step {step}, loss: {loss:.4f}")

Step 0, loss: 1.9656
Step 1, loss: 2.7474
Step 2, loss: 1.3948
Step 3, loss: 1.2125
Step 4, loss: 0.9893
Step 5, loss: 0.8803
Step 6, loss: 0.7976
Step 7, loss: 0.7341
Step 8, loss: 0.6984
Step 9, loss: 0.6671
Step 10, loss: 0.6333
Step 11, loss: 0.6090
Step 12, loss: 0.5795
Step 13, loss: 0.5510
Step 14, loss: 0.5296
Step 15, loss: 0.5212
Step 16, loss: 0.5175
Step 17, loss: 0.5015
Step 18, loss: 0.4828
Step 19, loss: 0.4710
Step 20, loss: 0.4668
Step 21, loss: 0.4603
Step 22, loss: 0.4512
Step 23, loss: 0.4462
Step 24, loss: 0.4409
Step 25, loss: 0.4373
Step 26, loss: 0.4318
Step 27, loss: 0.4259
Step 28, loss: 0.4221
Step 29, loss: 0.4201
Step 30, loss: 0.4153
Step 31, loss: 0.4123
Step 32, loss: 0.4087
Step 33, loss: 0.4064
Step 34, loss: 0.4015
Step 35, loss: 0.4004
Step 36, loss: 0.3972
Step 37, loss: 0.3968
Step 38, loss: 0.3930
Step 39, loss: 0.3919
Step 40, loss: 0.3909
Step 41, loss: 0.3880
Step 42, loss: 0.3868
Step 43, loss: 0.3844
Step 44, loss: 0.3842
Step 45, loss: 0.381

In [155]:
#nnx.display(model)
model.eval()
y = model(opendataset)
y = jnp.argmax(y, axis=1)
y = [cul_forward[item] for item in y]
df_y = pd.DataFrame(y, columns=['culture'])
df_openset = pd.read_csv("drive/MyDrive/test/openset_ans.csv", sep=";", encoding="windows-1251")
print(classification_report(df_openset, df_y, digits=5))

y = model(closeddataset)
y = jnp.argmax(y, axis=1)
y = [cul_forward[item] for item in y]
df_y = pd.DataFrame(y, columns=['culture'])
df_closedset = pd.read_csv("drive/MyDrive/closedtest/closedset_ans.csv", sep=";", encoding="windows-1251")
print(classification_report(df_closedset, df_y, digits=5))

                   precision    recall  f1-score   support

           залежь    0.98291   1.00000   0.99138       345
         зерновые    1.00000   0.99150   0.99573       353
         кукуруза    1.00000   1.00000   1.00000       332
многолетние травы    1.00000   1.00000   1.00000       190
            овощи    0.99194   1.00000   0.99595       123
              соя    0.99632   0.98188   0.98905       276

         accuracy                        0.99506      1619
        macro avg    0.99519   0.99556   0.99535      1619
     weighted avg    0.99512   0.99506   0.99506      1619

                   precision    recall  f1-score   support

           залежь    0.98799   1.00000   0.99396       329
         зерновые    1.00000   0.99427   0.99713       349
         кукуруза    0.99720   1.00000   0.99860       356
многолетние травы    1.00000   1.00000   1.00000       184
            овощи    1.00000   1.00000   1.00000       102
              соя    1.00000   0.99000   0.99497    